In [ ]:
import importlib.util
import subprocess
import sys

for package in ["transformers", "accelerate"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


In [ ]:
import ast
import html
import os
import random
import re
import time
import zipfile
from collections import Counter
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

SEED = 322
NUM_CLASSES = 5

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REPO_ROOT = Path.cwd()
WORK_DIR = REPO_ROOT
ARTIFACTS_DIR = WORK_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
def _columns_match(path, sep, required_columns):
    try:
        cols = set(pd.read_csv(path, sep=sep, nrows=0).columns)
        return set(required_columns).issubset(cols)
    except Exception:
        return False


def find_direct_csv(filename, sep, required_columns, roots):
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            candidates.append(direct)
        candidates.extend(root.rglob(filename))

    seen = set()
    for path in candidates:
        path = path.resolve()
        if path in seen:
            continue
        seen.add(path)
        if _columns_match(path, sep, required_columns):
            return path
    return None


def find_zip_with_data(roots):
    zip_candidates = []
    for root in roots:
        if not root.exists():
            continue
        zip_candidates.extend(root.glob("*.zip"))
        zip_candidates.extend(root.rglob("*.zip"))

    seen = set()
    for zip_path in zip_candidates:
        zip_path = zip_path.resolve()
        if zip_path in seen:
            continue
        seen.add(zip_path)
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = zf.namelist()
                basename_to_name = {Path(name).name: name for name in names}
                if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(basename_to_name):
                    return zip_path, basename_to_name
        except zipfile.BadZipFile:
            continue
    return None, None


search_roots = [REPO_ROOT]
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))

TRAIN_COLUMNS = ["id", "source", "title", "text", "publication_date", "target"]
TEST_COLUMNS = ["id", "source", "title", "text", "publication_date"]
SAMPLE_COLUMNS = ["id", "target"]

train_path = find_direct_csv("train.csv", "	", TRAIN_COLUMNS, search_roots)
test_path = find_direct_csv("test.csv", "	", TEST_COLUMNS, search_roots)
sample_path = find_direct_csv("sample_submission.csv", ",", SAMPLE_COLUMNS, search_roots)

if train_path is not None and test_path is not None and sample_path is not None:
    print("DATA_MODE: direct csv")
    print("train:", train_path)
    print("test:", test_path)
    print("sample:", sample_path)
    train_df = pd.read_csv(train_path, sep="	")
    test_df = pd.read_csv(test_path, sep="	")
    sample_df = pd.read_csv(sample_path)
else:
    zip_path, members = find_zip_with_data(search_roots)
    if zip_path is None:
        raise FileNotFoundError(
            "Не найдены train.csv/test.csv/sample_submission.csv ни напрямую, ни внутри zip-архива. "
            "Положите датасет соревнования в корень репозитория."
        )

    print("DATA_MODE: zip")
    print("zip:", zip_path)
    print("train member:", members["train.csv"])
    print("test member:", members["test.csv"])
    print("sample member:", members["sample_submission.csv"])

    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(members["train.csv"]) as f:
            train_df = pd.read_csv(f, sep="	")
        with zf.open(members["test.csv"]) as f:
            test_df = pd.read_csv(f, sep="	")
        with zf.open(members["sample_submission.csv"]) as f:
            sample_df = pd.read_csv(f)

print("train:", train_df.shape)
print("test:", test_df.shape)
print("sample:", sample_df.shape)

assert set(TRAIN_COLUMNS).issubset(train_df.columns)
assert set(TEST_COLUMNS).issubset(test_df.columns)
assert set(SAMPLE_COLUMNS).issubset(sample_df.columns)
assert set(sample_df["id"]) == set(test_df["id"])


## EDA

Целевая переменная хранится как строковое представление списка из пяти бинарных меток.  
Смотрим баланс классов, число меток на объект и распределение источников.


In [ ]:
def parse_target(value):
    if isinstance(value, list):
        parsed = value
    else:
        parsed = ast.literal_eval(str(value))
    assert len(parsed) == NUM_CLASSES
    return [int(x) for x in parsed]


y = np.vstack(train_df["target"].map(parse_target).values).astype(np.float32)
for class_id in range(NUM_CLASSES):
    train_df[f"target_{class_id}"] = y[:, class_id].astype(int)

label_cols = [f"target_{class_id}" for class_id in range(NUM_CLASSES)]
class_stats = pd.DataFrame(
    {
        "positive_count": y.sum(axis=0).astype(int),
        "positive_rate": y.mean(axis=0),
    },
    index=label_cols,
)

cardinality = pd.Series(y.sum(axis=1).astype(int)).value_counts().sort_index().to_frame("num_rows")

print("positive counts/rates:")
display(class_stats)
print("cardinality:")
display(cardinality)
print("sources train:")
display(train_df["source"].value_counts().to_frame("count"))
print("sources test:")
display(test_df["source"].value_counts().to_frame("count"))

display(train_df.head())
display(test_df.head())


In [ ]:
train_dates = pd.to_datetime(train_df["publication_date"], errors="coerce")
test_dates = pd.to_datetime(test_df["publication_date"], errors="coerce")

print("train date range:", train_dates.min(), train_dates.max())
print("test date range:", test_dates.min(), test_dates.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_dates.dt.to_period("M").value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Train by month")
test_dates.dt.to_period("M").value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Test by month")
plt.tight_layout()
plt.show()


## Текстовая очистка

Для transformer-модели не делаем лемматизацию, удаление стоп-слов или агрессивное упрощение текста.  
Оставляем смысловую лексику, числа и пунктуацию, но убираем HTML/XML-мусор, emoji и лишние пробелы.

Заголовок дублируется, потому что в новостях он часто несет основной сигнал класса.


In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")
BAD_CHARS_RE = re.compile(r"[^0-9a-zA-Zа-яА-ЯёЁ%.,!?;:()\-\s]")


def clean_text(text):
    text = str(text)
    text = html.unescape(text)
    text = TAG_RE.sub(" ", text)
    text = text.replace("CDATA", " ")
    text = BAD_CHARS_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def make_model_text(df):
    source = df["source"].fillna("unknown").astype(str)
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)
    return ("Источник: " + source + " [SEP] Заголовок: " + title + " [SEP] Заголовок: " + title + " [SEP] Текст: " + text).tolist()


train_texts = make_model_text(train_df)
test_texts = make_model_text(test_df)

lengths = pd.Series([len(text) for text in train_texts])
print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("example text:")
print(train_texts[0][:1000])


## Validation split и метрика

Локально оптимизируем hamming loss: долю неверных бинарных меток.  
Split стратифицируется по комбинации пяти классов, чтобы validation был похож на train по multilabel-структуре.


In [ ]:
def hamming_loss_np(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    return (y_true != y_pred).mean()


def tune_thresholds_for_hamming(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)

    thresholds = []
    class_losses = []

    for class_id in range(y_true.shape[1]):
        losses = []
        for threshold in grid:
            pred = (proba[:, class_id] >= threshold).astype(int)
            losses.append((y_true[:, class_id] != pred).mean())
        best_idx = int(np.argmin(losses))
        thresholds.append(float(grid[best_idx]))
        class_losses.append(float(losses[best_idx]))

    return np.array(thresholds), np.array(class_losses)


label_combo = ["".join(str(int(x)) for x in row) for row in y]
combo_counts = Counter(label_combo)
stratify_key = np.array([combo if combo_counts[combo] >= 2 else "rare" for combo in label_combo])

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.15,
    random_state=SEED,
    stratify=stratify_key,
)

print("train:", len(train_idx), "val:", len(val_idx))
print("val all-zero hamming loss:", hamming_loss_np(y[val_idx], np.zeros_like(y[val_idx])))


## Dataset и DataLoader

In [ ]:
MODEL_NAME = "ai-forever/ruRoberta-large"
MAX_LENGTH = 384
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EPOCHS = 4
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
NUM_WORKERS = 2

print("MODEL_NAME:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("EPOCHS:", EPOCHS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class NewsDataset(Dataset):
    def __init__(self, texts, targets=None):
        self.texts = list(texts)
        self.targets = targets

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in item.items()}

        if self.targets is not None:
            item["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)

        return item


train_dataset = NewsDataset([train_texts[i] for i in train_idx], y[train_idx])
val_dataset = NewsDataset([train_texts[i] for i in val_idx], y[val_idx])
test_dataset = NewsDataset(test_texts, None)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)


## Модель

In [ ]:
class RobertaForMultilabel(nn.Module):
    def __init__(self, model_name, num_classes=5, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0]

        logits = self.classifier(self.dropout(pooled))
        return logits


model = RobertaForMultilabel(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)

pos = torch.tensor(y[train_idx].sum(axis=0), dtype=torch.float32)
neg = torch.tensor(len(train_idx) - y[train_idx].sum(axis=0), dtype=torch.float32)
pos_weight = (neg / pos.clamp_min(1.0)).to(DEVICE)

print("pos_weight:", pos_weight.detach().cpu().numpy())
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Обучение

In [ ]:
def move_batch_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


@torch.no_grad()
def predict_proba(loader):
    model.eval()
    all_probs = []
    all_targets = []

    for batch in tqdm(loader, leave=False):
        batch = move_batch_to_device(batch)
        labels = batch.pop("labels", None)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(**batch)
            probs = torch.sigmoid(logits)

        all_probs.append(probs.detach().cpu().numpy())
        if labels is not None:
            all_targets.append(labels.detach().cpu().numpy())

    probs = np.vstack(all_probs)
    targets = np.vstack(all_targets) if all_targets else None
    return probs, targets


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
total_steps = EPOCHS * num_update_steps_per_epoch
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_loss = 1.0
best_thresholds = np.array([0.5] * NUM_CLASSES)
best_state_path = ARTIFACTS_DIR / "best_roberta_large.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    train_loss = 0.0
    start = time.time()

    progress = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(progress, 1):
        batch = move_batch_to_device(batch)
        labels = batch.pop("labels")

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(**batch)
            loss = criterion(logits, labels)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        train_loss += loss.item() * GRAD_ACCUM_STEPS * labels.size(0)

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        progress.set_postfix(loss=float(loss.item() * GRAD_ACCUM_STEPS))

    train_loss = train_loss / len(train_loader.dataset)

    val_proba, val_targets = predict_proba(val_loader)
    thresholds, class_losses = tune_thresholds_for_hamming(val_targets, val_proba)
    val_pred = (val_proba >= thresholds.reshape(1, -1)).astype(int)
    val_loss = hamming_loss_np(val_targets, val_pred)
    val_loss_05 = hamming_loss_np(val_targets, (val_proba >= 0.5).astype(int))

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_hamming_loss": val_loss,
        "val_hamming_loss_05": val_loss_05,
        "thresholds": thresholds.tolist(),
        "elapsed_min": (time.time() - start) / 60,
    }
    history.append(row)
    print(row)

    if val_loss < best_loss:
        best_loss = val_loss
        best_thresholds = thresholds.copy()
        torch.save(model.state_dict(), best_state_path)
        print("saved best:", best_state_path, "loss:", best_loss)

history_df = pd.DataFrame(history)
display(history_df)


## Validation report

In [ ]:
model.load_state_dict(torch.load(best_state_path, map_location=DEVICE))
model.eval()

val_proba, val_targets = predict_proba(val_loader)
val_pred = (val_proba >= best_thresholds.reshape(1, -1)).astype(int)

print("best hamming loss:", hamming_loss_np(val_targets, val_pred))
print("best thresholds:", best_thresholds)
print("true positive rates:", val_targets.mean(axis=0))
print("pred positive rates:", val_pred.mean(axis=0))
print(classification_report(val_targets, val_pred, zero_division=0))


## Inference и postprocessing

Postprocessing состоит в применении индивидуального threshold для каждого класса. Thresholds подобраны на validation split по hamming loss.


In [ ]:
test_proba, _ = predict_proba(test_loader)
test_pred = (test_proba >= best_thresholds.reshape(1, -1)).astype(int)

print("test positive rates:", test_pred.mean(axis=0))
print("test cardinality:")
print(pd.Series(test_pred.sum(axis=1)).value_counts().sort_index())


def format_target(row):
    return "[" + ",".join(str(int(x)) for x in row) + "]"


submission_df = sample_df.copy()
pred_by_id = dict(zip(test_df["id"].tolist(), [format_target(row) for row in test_pred]))
submission_df["target"] = submission_df["id"].map(pred_by_id)

assert submission_df["target"].notna().all()
submission_path = WORK_DIR / "sample_submission.csv"
submission_df.to_csv(submission_path, index=False)

print("saved:", submission_path)
print(submission_df.shape)
display(submission_df.head())
print(submission_df["target"].value_counts().head(20))


## Проверка файла сабмита

In [ ]:
check_df = pd.read_csv(WORK_DIR / "sample_submission.csv")

bad_rows = []
for idx, row in check_df.iterrows():
    try:
        value = ast.literal_eval(row["target"])
        if not isinstance(value, list) or len(value) != NUM_CLASSES:
            bad_rows.append((idx, row["target"], "bad_length"))
            continue
        if any(x not in [0, 1] for x in value):
            bad_rows.append((idx, row["target"], "bad_values"))
    except Exception as exc:
        bad_rows.append((idx, row["target"], str(exc)))

print("shape:", check_df.shape)
print("null target:", check_df["target"].isna().sum())
print("unique id:", check_df["id"].nunique())
print("bad rows:", len(bad_rows))
print(bad_rows[:5])
assert check_df.shape[0] == sample_df.shape[0]
assert check_df["id"].nunique() == sample_df.shape[0]
assert check_df["target"].notna().all()
assert not bad_rows
